# IBL Brain-wide Neuropixels: Peri-event Responses and Decoding

This notebook introduces two common analyses for spike-resolved behavioral recordings: peri-event time histograms (PETHs) aligned to a task event and cross-validated logistic decoding of a binary trial label. It runs entirely on synthetic trials and spike times by default. A guarded ONE/Alyx cell provides a starting point for authorized public-data exploration.

A decoder demonstrates information available in the chosen features under this validation scheme. It does not identify a neural mechanism, establish causal contribution, or necessarily generalize across animals, sessions, task conditions, or recording drift.

## Quick start

For the default offline path, run the minimal-install cell and then run the synthetic trial and spike cells from top to bottom. You should see a peri-event response plot and a cross-validated synthetic logistic-decoder score. The later ONE/Alyx cells are optional advanced previews and do not supply data to or alter the synthetic analysis.

## Prerequisites

- Python 3.9+, NumPy, Matplotlib, and basic spike-train/binning concepts
- Familiarity with trial events, leakage, and cross-validation
- Optional live access needs internet connectivity; ONE may prompt for configuration or require acceptance of current data-access conditions

## Setup

Install the minimal packages below for the default offline tutorial. No electrophysiology dataset is downloaded by default.

In [ ]:
%pip install -q numpy matplotlib scikit-learn

### Optional installation for advanced live ONE/Alyx previews

Install this client only if you will enable the advanced preview cells below. The default synthetic analysis does not import it.

In [ ]:
%pip install -q one-api

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

rng = np.random.default_rng(18)

## Advanced preview (optional): discover an IBL session through ONE/Alyx

Live catalog queries and dataset availability change over time. Set `USE_LIVE_ONE=True` to make a small metadata query, then choose a session and datasets deliberately. Downloading spike sorting output can be large, and probe insertions, quality labels, and trial definitions must be recorded in a real analysis.

In [ ]:
USE_LIVE_ONE = False

if USE_LIVE_ONE:
    from one.api import ONE
    one = ONE(base_url='https://openalyx.internationalbrainlab.org', silent=True)
    eids = one.search(subject='KS023', date='2019-12-10')
    if not eids:
        raise RuntimeError('No matching session found; query a current subject/date in ONE.')
    eid = eids[0]
    print('Example session:', eid)
    print(one.list_datasets(eid, filename='*trials*')[:10])
else:
    print('Live ONE access is disabled; using synthetic trials and spikes.')


## Advanced preview (optional): load selected trials and a small spike object through ONE

This advanced preview does not feed the synthetic analysis below. First use the discovery cell to inspect a session. Then explicitly set `LIVE_EID` to a selected experiment ID and `LIVE_SPIKES_COLLECTION` to a collection returned by `one.list_collections(LIVE_EID, filename='spikes.times.npy')`. Both objects are loaded only when the flag is enabled. This can still download sizeable files; begin with one probe collection, retain the dataset version/provenance, and apply cluster-quality and trial-exclusion criteria before scientific analysis.

In [ ]:
LOAD_LIVE_ONE_OBJECTS = False
LIVE_EID = None  # selected experiment UUID from the discovery query
LIVE_SPIKES_COLLECTION = None  # selected collection containing spikes.times.npy

if LOAD_LIVE_ONE_OBJECTS:
    if not USE_LIVE_ONE or 'one' not in globals():
        raise RuntimeError('Enable and run the ONE discovery cell first.')
    if not isinstance(LIVE_EID, str) or not LIVE_EID:
        raise ValueError('Set LIVE_EID to an explicitly selected experiment UUID.')
    if not isinstance(LIVE_SPIKES_COLLECTION, str) or not LIVE_SPIKES_COLLECTION:
        raise ValueError('Set LIVE_SPIKES_COLLECTION after inspecting one.list_collections().')

    trial_datasets = one.list_datasets(LIVE_EID, filename='*trials*')
    spike_datasets = one.list_datasets(
        LIVE_EID, collection=LIVE_SPIKES_COLLECTION, filename='spikes.*'
    )
    if not trial_datasets:
        raise FileNotFoundError('No trials datasets found for LIVE_EID; choose another session.')
    if not spike_datasets:
        raise FileNotFoundError('No spike datasets found in LIVE_SPIKES_COLLECTION.')

    live_trials = one.load_object(LIVE_EID, 'trials')
    live_spikes = one.load_object(
        LIVE_EID, 'spikes', collection=LIVE_SPIKES_COLLECTION,
        attribute=['times', 'clusters'],
    )
    required_trial_fields = {'goCue_times', 'choice'}
    missing_trials = required_trial_fields.difference(live_trials.keys())
    required_spike_fields = {'times', 'clusters'}
    missing_spikes = required_spike_fields.difference(live_spikes.keys())
    if missing_trials or missing_spikes:
        raise KeyError(f'Missing trial fields: {missing_trials}; missing spike fields: {missing_spikes}')
    print(f'Loaded {len(live_trials.choice)} trials and {len(live_spikes.times)} spike times.')
else:
    print('Live ONE object loading is disabled; no trials or spike files will be downloaded.')


## Simulate trials and spike times

We create 160 trial-aligned events and spikes from one illustrative unit. On synthetic trials, the latent binary label changes the post-event rate. This lets the PETH and decoder code execute, but is not a model of any particular IBL neuron or task variable.

In [ ]:
n_trials = 160
event_times = np.arange(n_trials, dtype=float) * 3.0 + 1.0
choice_right = rng.integers(0, 2, n_trials)

spike_times = []
for event, label in zip(event_times, choice_right):
    # Baseline spikes and a label-dependent 0–300 ms response.
    baseline = event + rng.uniform(-0.5, 0, rng.poisson(4))
    response_rate = 5 + 18 * label
    response = event + rng.uniform(0, 0.3, rng.poisson(response_rate * 0.3))
    spike_times.extend(np.r_[baseline, response])
spike_times = np.sort(np.asarray(spike_times))
print(f'{n_trials} synthetic trials; {len(spike_times)} synthetic spikes')

## Peri-event time histogram

For each event, bin spikes relative to the event, average across trials, then divide by bin width to obtain spikes/s. The uncertainty band below is a trial-level standard error, not uncertainty over animals or recording sessions.

In [ ]:
bin_width = 0.025
bins = np.arange(-0.5, 0.501, bin_width)
centers = (bins[:-1] + bins[1:]) / 2

def event_counts(times, events, bin_edges):
    return np.vstack([np.histogram(times - event, bins=bin_edges)[0] for event in events])

counts = event_counts(spike_times, event_times, bins)
rate = counts.mean(axis=0) / bin_width
sem = counts.std(axis=0, ddof=1) / np.sqrt(n_trials) / bin_width
plt.plot(centers, rate, color='black')
plt.fill_between(centers, rate - sem, rate + sem, color='gray', alpha=0.35)
plt.axvline(0, color='tab:red', linestyle='--', label='event')
plt.xlabel('Time from event (s)'); plt.ylabel('Rate (spikes/s)'); plt.legend()
plt.title('Synthetic unit PETH; trial-level mean ± SEM')

## Trial features and a leakage-safe logistic decoder

We count spikes in a pre-event baseline window and two post-event windows. The classifier uses stratified folds so that class balance is represented in each split. For real IBL data, split by session/animal where generalization is the question, and make all preprocessing choices inside each training fold.

In [ ]:
feature_windows = [(-0.4, 0.0), (0.0, 0.15), (0.15, 0.3)]
X = np.column_stack([
    [np.sum((spike_times >= event + start) & (spike_times < event + stop))
     for event in event_times]
    for start, stop in feature_windows
])
y = choice_right
print('Feature matrix:', X.shape)

decoder = LogisticRegression(max_iter=2_000, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(decoder, X, y, cv=cv, scoring='roc_auc')
print(f'Cross-validated ROC-AUC: {scores.mean():.3f} ± {scores.std(ddof=1):.3f} (fold SD)')
print('This score is expected to be above chance because the synthetic generator encodes the label post-event.')

## Good practice for a brain-wide analysis

Specify event definitions, trial inclusion, cluster-quality criteria, probe/region mapping, binning, and the exact cross-validation grouping before inspecting results. Assess chance with label permutations that respect session structure. When screening many units or regions, account for multiplicity and report effects and uncertainty, not only thresholded p-values. Population decoders should never mix test-trial information into feature normalization or unit selection.

## References

- International Brain Laboratory et al. (2025). A brain-wide map of neural activity during complex behaviour. *Nature*, 638, 481–491. https://doi.org/10.1038/s41586-025-08639-3
- International Brain Laboratory et al. (2021). Standardized and reproducible measurement of decision-making in mice. *eLife*, 10, e63711. https://doi.org/10.7554/eLife.63711
- ONE API documentation. https://int-brain-lab.github.io/ONE/
- IBL data access documentation. https://docs.internationalbrainlab.org/

## License

This notebook is released under the repository's license. Its default data are synthetic. IBL data, software, and metadata remain subject to their current licenses, data-use terms, and citation requirements.